# EnergIA · Evaluación de los detectores de anomalías
### Comparación de LOF, COF, CBLOF e Isolation Forest con anomalías sintéticas

Este notebook solo evalúa; no modifica los modelos del sistema.

Para calcular precisión, recall, F1 y FPR se necesita saber qué horas son anómalas, y los datos medidos no traen esa etiqueta. Por eso, en la semana de prueba (la misma del notebook de entrenamiento) se alteran algunas horas con fallas típicas y se mide cuántas detecta cada modelo. Las horas alteradas son las positivas; el resto, las negativas.

**Procedimiento**
1. Carga, limpieza, minutos y tabla horaria, igual que en el notebook de entrenamiento.
2. División cronológica: entrenamiento con las horas anteriores a la semana de prueba.
3. Inyección del 5 % de horas anómalas por aparato, de cuatro tipos.
4. Entrenamiento de cada modelo por aparato con el umbral del sistema (percentil 1 del entrenamiento).
5. Diez repeticiones con distintas horas alteradas; se promedian las métricas.

**Ejecución en Google Colab:** con `lecturas_2s_1.csv` en Google Drive, revisar `RUTA_DRIVE` y ejecutar todas las celdas. Al final se descarga `resultados_anomalias.zip` con las tablas y las figuras.

## 1. Configuración
- `RUTA_DRIVE`: ubicación del archivo de datos en Google Drive.
- `TARIFA_COP_KWH`: valor del kWh en pesos; solo se usa para mostrar costos.

In [ ]:
# Parámetros
RUTA_DRIVE     = '/content/drive/MyDrive/lecturas_2s_1.csv'   # ubicación del archivo en Google Drive
TARIFA_COP_KWH = 850
SEMILLA        = 0

# Rango de medición del PZEM-004T v3
RANGO = {'voltaje': (80, 260), 'corriente': (0, 100), 'potencia': (0, 23000)}

# Librerías
import warnings; warnings.filterwarnings('ignore')
import json, os, shutil, time
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import sklearn, joblib
from IPython.display import display, Markdown

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_columns', 30)

# Estilo de las gráficas
ORDEN = ['Nevera1', 'Nevera2', 'TVyDeco', 'PC', 'Router']
COLOR = {'Nevera1': '#2a78d6', 'Nevera2': '#eb6834', 'TVyDeco': '#1baf7a', 'PC': '#6250d6', 'Router': '#e34948'}
GRIS, TINTA, TINTA2 = '#d9d8d4', '#0b0b0b', '#52514e'
plt.rcParams.update({
    'figure.figsize': (12, 4.5), 'figure.dpi': 110, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.edgecolor': GRIS, 'axes.labelcolor': TINTA2,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': TINTA, 'axes.labelsize': 10.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#ecebe8', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': TINTA2, 'ytick.color': TINTA2, 'xtick.labelsize': 9.5, 'ytick.labelsize': 9.5,
    'legend.frameon': False, 'legend.fontsize': 10, 'lines.linewidth': 2,
})
fmt_cop = lambda x: f'${x:,.0f}'.replace(',', '.')

print(f'scikit-learn {sklearn.__version__} · pandas {pd.__version__} · numpy {np.__version__}')
print('IMPORTANTE: instala esta misma versión de scikit-learn en la Raspberry:')
print(f'  /home/luisool/venv/bin/pip install scikit-learn=={sklearn.__version__}')

## 2. Variables de los modelos (`variables.py`)
Esta celda escribe `variables.py`, el mismo archivo que usa el backend, de modo que los modelos reciben en la Raspberry las variables con las que se entrenaron.

In [ ]:
%%writefile variables.py
"""
variables.py · Variables (features) de los modelos de EnergIA.

Este mismo archivo lo usan el notebook de entrenamiento (Colab) y el backend
(Raspberry). Si el backend calcula las variables de otra forma, el modelo
predice valores sin sentido sin dar ningún error; por eso el cálculo vive en
un solo lugar.

Entrada común: una tabla HORARIA con una fila por aparato y hora:
    fecha_hora  (inicio de la hora, hora de Colombia, sin zona horaria)
    aparato     (Nevera1, Nevera2, Router, PC, TVyDeco)
    kwh         (energía consumida en esa hora)
"""

import numpy as np
import pandas as pd

APARATOS = ['Nevera1', 'Nevera2', 'Router', 'PC', 'TVyDeco']
NODO_A_APARATO = {'nodo_1': 'Nevera1', 'nodo_2': 'Nevera2', 'nodo_3': 'Router',
                  'nodo_4': 'PC', 'nodo_5': 'TVyDeco'}

# Festivos de Colombia (Ley Emiliani) 2026 y 2027
FESTIVOS = {pd.Timestamp(d).date() for d in [
    '2026-01-01', '2026-01-12', '2026-03-23', '2026-04-02', '2026-04-03', '2026-05-01',
    '2026-05-18', '2026-06-08', '2026-06-15', '2026-06-29', '2026-07-20', '2026-08-07',
    '2026-08-17', '2026-10-12', '2026-11-02', '2026-11-16', '2026-12-08', '2026-12-25',
    '2027-01-01', '2027-01-11', '2027-03-22', '2027-03-25', '2027-03-26', '2027-05-01',
    '2027-05-10', '2027-05-31', '2027-06-07', '2027-07-05', '2027-07-20', '2027-08-07',
    '2027-08-16', '2027-10-18', '2027-11-01', '2027-11-15', '2027-12-08', '2027-12-25',
]}

# Variables del modelo de predicción, en este orden exacto
VARIABLES_PREDICCION = [
    'hora_sin', 'hora_cos', 'dia_sin', 'dia_cos', 'fin_de_semana_o_festivo',
    'lag24', 'lag48', 'lag168', 'media_dia_anterior', 'media_misma_hora_7d',
    *[f'es_{a}' for a in APARATOS],
]

# Variables del detector de anomalías (por aparato), en este orden exacto
VARIABLES_ANOMALIAS = ['kwh', 'pct_encendido', 'arranques', 'p_encendido_media', 'hora_sin', 'hora_cos']

UMBRAL_ENCENDIDO_W = 2.4   # 0,02 A x 120 V: el mismo umbral de los nodos


def calendario(fechas: pd.Series) -> pd.DataFrame:
    """Hora y día codificados de forma cíclica (las 23 h quedan cerca de las 0 h)."""
    f = pd.to_datetime(fechas)
    hora = f.dt.hour
    dia = f.dt.dayofweek
    festivo = f.dt.date.isin(FESTIVOS)
    return pd.DataFrame({
        'hora_sin': np.sin(2 * np.pi * hora / 24),
        'hora_cos': np.cos(2 * np.pi * hora / 24),
        'dia_sin': np.sin(2 * np.pi * dia / 7),
        'dia_cos': np.cos(2 * np.pi * dia / 7),
        'fin_de_semana_o_festivo': ((dia >= 5) | festivo).astype(int),
    }, index=fechas.index)


def variables_prediccion(horaria: pd.DataFrame) -> pd.DataFrame:
    """Agrega las variables de predicción a la tabla horaria.

    Todas las variables de consumo miran al menos 24 horas atrás: así, con los
    datos hasta "ahora", se pueden predecir de una vez las 24 horas siguientes.
    Las filas sin historia suficiente quedan con NaN (se descartan al entrenar).
    """
    h = horaria.sort_values(['aparato', 'fecha_hora']).copy()
    # Rejilla completa de horas por aparato, para que "24 filas atrás" sea "24 horas atrás"
    partes = []
    for a, g in h.groupby('aparato', sort=False):
        horas = pd.date_range(g.fecha_hora.min(), g.fecha_hora.max(), freq='h')
        g = g.set_index('fecha_hora').reindex(horas)
        g.index.name = 'fecha_hora'
        g['aparato'] = a
        partes.append(g.reset_index())
    h = pd.concat(partes, ignore_index=True)

    k = h.groupby('aparato', sort=False)['kwh']
    for lag in (24, 48, 168):
        h[f'lag{lag}'] = k.shift(lag)
    # Promedio de las 24 horas que terminan 24 horas antes
    h['media_dia_anterior'] = k.transform(lambda s: s.shift(24).rolling(24, min_periods=18).mean())
    # Promedio de la misma hora en los 7 días anteriores
    h['media_misma_hora_7d'] = sum(k.shift(24 * d) for d in range(1, 8)) / 7

    h = pd.concat([h, calendario(h['fecha_hora'])], axis=1)
    for a in APARATOS:
        h[f'es_{a}'] = (h['aparato'] == a).astype(int)
    return h


HORIZONTE_HORAS = 168      # predicción semanal: los próximos 7 días
DIAS_HISTORIA = 9          # historia que necesita cada bloque (lag168 + promedio de 7 días)


def prediccion_recursiva(modelo, historia: pd.DataFrame, inicio, horas: int = HORIZONTE_HORAS) -> pd.DataFrame:
    """Predice `horas` horas desde `inicio` (inclusive) por bloques de 24 horas.

    Cada bloque se predice con las variables de la historia; luego sus predicciones
    se agregan a la historia como si fueran consumo medido y se calcula el bloque
    siguiente. Así el modelo de 24 horas llega a los 7 días.

    historia: tabla horaria (fecha_hora, aparato, kwh) con datos ANTERIORES a `inicio`.
    Devuelve: fecha_hora, aparato, kwh_pred y dia_horizonte (1 = primeras 24 horas).
    """
    inicio = pd.Timestamp(inicio)
    h = historia.loc[(historia.fecha_hora < inicio) &
                     (historia.fecha_hora >= inicio - pd.Timedelta(days=DIAS_HISTORIA)),
                     ['fecha_hora', 'aparato', 'kwh']]
    bloques = []
    for desde in range(0, horas, 24):
        ini = inicio + pd.Timedelta(hours=desde)
        n = min(24, horas - desde)
        futuro = pd.DataFrame([(ini + pd.Timedelta(hours=k), a, np.nan) for a in APARATOS for k in range(n)],
                              columns=['fecha_hora', 'aparato', 'kwh'])
        t = variables_prediccion(pd.concat([h, futuro], ignore_index=True))
        t = t[(t.fecha_hora >= ini) & (t.fecha_hora < ini + pd.Timedelta(hours=n))]
        pred = np.clip(modelo.predict(t[VARIABLES_PREDICCION]), 0, None)
        bloque = t[['fecha_hora', 'aparato']].assign(kwh_pred=pred, dia_horizonte=desde // 24 + 1)
        bloques.append(bloque)
        h = pd.concat([h, bloque.rename(columns={'kwh_pred': 'kwh'})[['fecha_hora', 'aparato', 'kwh']]],
                      ignore_index=True)
    return pd.concat(bloques, ignore_index=True)


def tabla_horaria_desde_minutos(minutos: pd.DataFrame) -> pd.DataFrame:
    """De una tabla por minuto (fecha_hora, aparato, potencia en W, voltaje)
    a la tabla horaria con las variables que usan los dos modelos."""
    m = minutos.copy()
    m['hora_ini'] = m['fecha_hora'].dt.floor('h')
    m['encendido'] = m['potencia'] >= UMBRAL_ENCENDIDO_W
    m = m.sort_values(['aparato', 'fecha_hora'])
    m['arranque'] = m.groupby('aparato')['encendido'].transform(lambda s: s & ~s.shift(1, fill_value=False))
    g = m.groupby(['aparato', 'hora_ini'])
    hor = pd.DataFrame({
        'kwh': g['potencia'].sum() / 60 / 1000,          # W·min -> kWh
        'minutos': g['potencia'].size(),
        'pct_encendido': g['encendido'].mean() * 100,
        'arranques': g['arranque'].sum(),
        'p_encendido_media': g.apply(lambda x: x.loc[x.encendido, 'potencia'].mean() if x.encendido.any() else 0.0),
        'voltaje_medio': g['voltaje'].mean(),
    }).reset_index().rename(columns={'hora_ini': 'fecha_hora'})
    return hor


def variables_anomalias(horaria: pd.DataFrame) -> pd.DataFrame:
    """Variables del detector de anomalías: una fila por aparato y hora."""
    h = horaria.copy()
    cal = calendario(h['fecha_hora'])
    h['hora_sin'], h['hora_cos'] = cal['hora_sin'], cal['hora_cos']
    return h

In [ ]:
import importlib, variables
importlib.reload(variables)
from variables import (APARATOS, NODO_A_APARATO, VARIABLES_PREDICCION, VARIABLES_ANOMALIAS,
                       variables_prediccion, tabla_horaria_desde_minutos, variables_anomalias)
print('Variables de predicción:', VARIABLES_PREDICCION)
print('Variables de anomalías: ', VARIABLES_ANOMALIAS)

## 3. Carga de los datos

In [ ]:
RUTA = 'lecturas_2s_1.csv'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA = RUTA_DRIVE
except ImportError:
    print('No estás en Colab: se usa la ruta local', RUTA)

inicio_carga = time.time()
df = pd.read_csv(RUTA, usecols=['fecha_hora_local', 'nodo', 'voltaje', 'corriente', 'potencia'],
                 dtype={'nodo': 'category'})
df['fecha_hora_local'] = pd.to_datetime(df['fecha_hora_local'], format='mixed')
df['aparato'] = df['nodo'].map(NODO_A_APARATO).astype('category')
print(f'{len(df):,} lecturas cargadas en {time.time() - inicio_carga:.0f} s')
print('Periodo:', df.fecha_hora_local.min(), '→', df.fecha_hora_local.max())
display(df.groupby('aparato', observed=True).size().rename('lecturas').to_frame())

## 4. Preparación

### 4.1 Lecturas fuera del rango del sensor
El PZEM-004T v3 mide de 80 a 260 V, de 0 a 100 A y de 0 a 23 kW; las lecturas por fuera son errores del sensor y se eliminan.

In [ ]:
fuera = np.zeros(len(df), bool)
for var, (lo, hi) in RANGO.items():
    fuera |= ~df[var].between(lo, hi).to_numpy()
print(f'Lecturas fuera de rango eliminadas: {fuera.sum()}')
display(df.loc[fuera, ['fecha_hora_local', 'aparato', 'voltaje', 'corriente', 'potencia']])
df = df.loc[~fuera]

### 4.2 De lecturas cada 2 segundos a minutos
Se promedia por minuto y por aparato. Los nodos no envían lecturas con el aparato apagado (menos de 0,02 A), así que se completa la rejilla de minutos del periodo y los minutos sin lecturas se registran como 0 W. Sin este paso, el promedio solo contaría los periodos encendidos.

El voltaje de un minuto sin lectura se toma del promedio de los otros nodos en ese minuto, porque todos comparten la red.

In [ ]:
df['minuto'] = df.fecha_hora_local.dt.floor('min')
m = (df.groupby(['aparato', 'minuto'], observed=True)[['voltaje', 'corriente', 'potencia']]
       .mean().reset_index())

minutos = pd.date_range(m.minuto.min(), m.minuto.max(), freq='min')
rejilla = pd.MultiIndex.from_product([APARATOS, minutos], names=['aparato', 'minuto'])
m = m.set_index(['aparato', 'minuto']).reindex(rejilla).reset_index()
m['potencia'] = m['potencia'].fillna(0.0)
m['corriente'] = m['corriente'].fillna(0.0)
v_red = m.groupby('minuto')['voltaje'].transform('mean')
m['voltaje'] = m['voltaje'].fillna(v_red).interpolate(limit_direction='both')
m = m.rename(columns={'minuto': 'fecha_hora'})
del df

print(f'{len(m):,} filas por minuto ({len(minutos):,} minutos x {len(APARATOS)} aparatos)')
resumen = m.groupby('aparato').agg(potencia_media_W=('potencia', 'mean'),
                                   pct_encendido=('potencia', lambda p: 100 * (p >= 2.4).mean()))
resumen['kWh_mes'] = resumen.potencia_media_W * 720 / 1000
resumen['COP_mes'] = resumen.kWh_mes * TARIFA_COP_KWH
display(resumen.reindex(ORDEN).round(1))
print(f'TOTAL: {resumen.kWh_mes.sum():.0f} kWh/mes ≈ {fmt_cop(resumen.COP_mes.sum())} COP/mes')

### 4.3 Tabla horaria
Para cada aparato y hora se calculan la energía (kWh), el porcentaje del tiempo encendido, el número de arranques, la potencia media encendido y el voltaje medio. Se descartan la primera y la última hora si están incompletas.

In [ ]:
horaria = tabla_horaria_desde_minutos(m)
horaria = horaria[horaria.minutos == 60].drop(columns='minutos').reset_index(drop=True)
print(f'{len(horaria):,} filas horarias · {horaria.fecha_hora.nunique():,} horas · '
      f'{horaria.fecha_hora.dt.normalize().nunique()} días')

perfil = horaria.assign(hora=horaria.fecha_hora.dt.hour).pivot_table(index='hora', columns='aparato', values='kwh')
fig, ax = plt.subplots(figsize=(12, 4))
for a in ORDEN:
    ax.plot(perfil.index, perfil[a] * 1000, color=COLOR[a], marker='o', ms=3, label=a)
ax.set_xticks(range(24)); ax.set_xlabel('Hora del día'); ax.set_ylabel('Wh por hora (promedio)')
ax.legend(ncol=5, loc='upper left'); ax.set_title('Consumo promedio por hora del día')
plt.tight_layout(); plt.show()

## 5. Entrenamiento y prueba
La semana de prueba es la última semana completa del conjunto, la misma del notebook de entrenamiento. Los modelos se entrenan con las horas anteriores.

In [ ]:
ha = variables_anomalias(horaria).dropna(subset=VARIABLES_ANOMALIAS).reset_index(drop=True)
fin_datos = horaria.fecha_hora.max() + pd.Timedelta(hours=1)
INICIO_PRUEBA = (fin_datos - pd.Timedelta(days=7)).floor('D')
FIN_PRUEBA = INICIO_PRUEBA + pd.Timedelta(days=7)
entreno = ha[ha.fecha_hora < INICIO_PRUEBA].reset_index(drop=True)
prueba = ha[(ha.fecha_hora >= INICIO_PRUEBA) & (ha.fecha_hora < FIN_PRUEBA)].reset_index(drop=True)
print(f'Entrenamiento: {entreno.fecha_hora.min():%d-%b %H:%M} a {entreno.fecha_hora.max():%d-%b %H:%M} · {len(entreno):,} horas-aparato')
print(f'Prueba:        {INICIO_PRUEBA:%d-%b} a {FIN_PRUEBA - pd.Timedelta(hours=1):%d-%b %H:%M} · {len(prueba):,} horas-aparato')
print('Variables:', VARIABLES_ANOMALIAS)

## 6. Anomalías sintéticas
En cada repetición se altera al azar el 5 % de las horas de prueba de cada aparato con uno de cuatro tipos de falla, manteniendo la relación entre energía, potencia y tiempo encendido.

| Tipo | Simula | Alteración |
|---|---|---|
| `consumo_alto` | Consumo muy superior al normal | Energía ×2 a ×3; si estaba apagado, encendido entre el 40 % y el 90 % de la hora |
| `apagado` | Aparato desconectado o dañado | Energía, tiempo encendido, arranques y potencia en 0 (solo en horas encendido) |
| `ciclado` | Arranques y paradas anormales | Arranques ×3 + 6 a 12; tiempo encendido ×0,85 |
| `potencia` | Potencia distinta de la habitual | Potencia encendida ×0,6 o ×1,6 |

In [ ]:
TASA_ANOMALIAS = 0.05
REPETICIONES = 10
TIPOS = ['consumo_alto', 'apagado', 'ciclado', 'potencia']
p_tipica = entreno[entreno.p_encendido_media > 0].groupby('aparato').p_encendido_media.median()


def ajustar_kwh(f):
    f['kwh'] = f['p_encendido_media'] * f['pct_encendido'] / 100 / 1000
    return f


def inyectar(prueba, semilla):
    rng = np.random.default_rng(semilla)
    datos = prueba.copy()
    datos['anomala'] = False
    datos['tipo'] = 'normal'
    for a, g in datos.groupby('aparato'):
        n = max(1, round(TASA_ANOMALIAS * len(g)))
        elegidas = rng.choice(g.index.to_numpy(), size=n, replace=False)
        tipos = rng.permutation(np.resize(TIPOS, n))
        for i, tipo in zip(elegidas, tipos):
            f = datos.loc[i].copy()
            encendido = f.pct_encendido > 20
            if tipo == 'apagado' and not encendido:
                tipo = 'consumo_alto'
            if tipo == 'potencia' and f.pct_encendido == 0:
                tipo = 'consumo_alto'
            if tipo == 'consumo_alto':
                if f.pct_encendido > 0:
                    objetivo = f.kwh * rng.uniform(2, 3)
                    f['p_encendido_media'] *= rng.uniform(1.0, 1.5)
                    f['pct_encendido'] = min(100.0, objetivo * 1000 / f.p_encendido_media * 100)
                else:
                    f['p_encendido_media'] = p_tipica.get(a, 50.0)
                    f['pct_encendido'] = rng.uniform(40, 90)
                    f['arranques'] = max(1, f.arranques)
                f = ajustar_kwh(f)
            elif tipo == 'apagado':
                f[['kwh', 'pct_encendido', 'arranques', 'p_encendido_media']] = 0.0
            elif tipo == 'ciclado':
                f['arranques'] = f.arranques * 3 + rng.integers(6, 13)
                if f.pct_encendido == 0:
                    f['p_encendido_media'] = p_tipica.get(a, 50.0)
                    f['pct_encendido'] = rng.uniform(20, 50)
                else:
                    f['pct_encendido'] *= 0.85
                f = ajustar_kwh(f)
            elif tipo == 'potencia':
                f['p_encendido_media'] *= rng.choice([0.6, 1.6])
                f = ajustar_kwh(f)
            f['anomala'], f['tipo'] = True, tipo
            datos.loc[i] = f
    return datos


ejemplo = inyectar(prueba, 0)
print(f'Horas anómalas por repetición: {ejemplo.anomala.sum()} de {len(ejemplo)} ({100 * ejemplo.anomala.mean():.1f} %)')
display(pd.crosstab(ejemplo.aparato, ejemplo.tipo).reindex(ORDEN))

## 7. Modelos
Todos se entrenan por aparato, con las mismas variables y el mismo umbral del sistema. Las variables se estandarizan con la media y la desviación del entrenamiento, porque LOF, COF y CBLOF se basan en distancias.

| Modelo | Principio | Parámetros |
|---|---|---|
| LOF (Local Outlier Factor) | Densidad local del punto frente a la de sus vecinos | 20 vecinos |
| COF (Connectivity-based Outlier Factor) | Distancia de encadenamiento con sus vecinos | 20 vecinos |
| CBLOF (Cluster-based Local Outlier Factor) | Distancia al grupo grande más cercano (k-medias) | 8 grupos, α = 0,9, β = 5 |
| Isolation Forest (desplegado) | Número de particiones aleatorias necesarias para aislar el punto | 200 árboles |

COF y CBLOF se implementan en el notebook según Tang et al. (2002) y He et al. (2003).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

PERCENTIL_UMBRAL = 1.0   # percentil 1 del entrenamiento, como en el sistema
K_VECINOS, N_GRUPOS, ALFA, BETA = 20, 8, 0.9, 5
EPS = 1e-9


def _ac_dist(punto, vecinos):
    """Distancia de encadenamiento promedio (COF): camino SBN desde el punto por sus vecinos."""
    S = np.vstack([punto, vecinos])
    D = np.linalg.norm(S[:, None, :] - S[None, :, :], axis=2)
    k = len(vecinos)
    dentro = [0]
    fuera = list(range(1, k + 1))
    costos = []
    for _ in range(k):
        sub = D[np.ix_(dentro, fuera)]
        j = np.unravel_index(np.argmin(sub), sub.shape)[1]
        costos.append(sub.min(axis=0)[j])
        dentro.append(fuera.pop(j))
    pesos = np.array([2 * (k + 1 - i) / (k * (k + 1)) for i in range(1, k + 1)])
    return float(np.dot(pesos, costos))


class COF:
    def fit(self, X):
        self.X = X
        self.nn = NearestNeighbors(n_neighbors=K_VECINOS + 1).fit(X)
        idx = self.nn.kneighbors(X, return_distance=False)[:, 1:]
        self.ac = np.array([_ac_dist(X[i], X[idx[i]]) for i in range(len(X))])
        self.puntaje_entreno = self.ac / (self.ac[idx].mean(axis=1) + EPS)
        return self

    def puntaje(self, Z):
        idx = self.nn.kneighbors(Z, n_neighbors=K_VECINOS, return_distance=False)
        ac = np.array([_ac_dist(Z[i], self.X[idx[i]]) for i in range(len(Z))])
        return ac / (self.ac[idx].mean(axis=1) + EPS)


class CBLOF:
    def fit(self, X):
        self.km = KMeans(n_clusters=N_GRUPOS, n_init=10, random_state=SEMILLA).fit(X)
        tam = np.bincount(self.km.labels_, minlength=N_GRUPOS)
        orden = np.argsort(tam)[::-1]
        t = tam[orden]
        b = len(t) - 1
        for i in range(len(t) - 1):
            if t[:i + 1].sum() >= ALFA * len(X) or (t[i + 1] > 0 and t[i] / t[i + 1] >= BETA):
                b = i
                break
        self.grandes = orden[:b + 1]
        self.puntaje_entreno = self.puntaje(X)
        return self

    def puntaje(self, Z):
        c = self.km.cluster_centers_
        grupo = self.km.predict(Z)
        d_propio = np.linalg.norm(Z - c[grupo], axis=1)
        d_grande = np.linalg.norm(Z[:, None, :] - c[self.grandes][None, :, :], axis=2).min(axis=1)
        return np.where(np.isin(grupo, self.grandes), d_propio, d_grande)


class LOF:
    def fit(self, X):
        self.m = LocalOutlierFactor(n_neighbors=K_VECINOS, novelty=True).fit(X)
        self.puntaje_entreno = -self.m.negative_outlier_factor_
        return self

    def puntaje(self, Z):
        return -self.m.score_samples(Z)


class IF:
    def fit(self, X):
        self.m = IsolationForest(n_estimators=200, contamination='auto', random_state=SEMILLA).fit(X)
        self.puntaje_entreno = -self.m.decision_function(X)
        return self

    def puntaje(self, Z):
        return -self.m.decision_function(Z)


MODELOS = {'LOF': LOF, 'COF': COF, 'CBLOF': CBLOF, 'Isolation Forest': IF}

# Un entrenamiento por aparato y modelo (igual en todas las repeticiones)
entrenados = {}
t0 = time.time()
for a in ORDEN:
    Xe = entreno.loc[entreno.aparato == a, VARIABLES_ANOMALIAS].to_numpy(float)
    esc = StandardScaler().fit(Xe)
    for nombre, Clase in MODELOS.items():
        m = Clase().fit(esc.transform(Xe))
        entrenados[(a, nombre)] = {'modelo': m, 'escala': esc,
                                  'umbral': float(np.percentile(m.puntaje_entreno, 100 - PERCENTIL_UMBRAL))}
print(f'{len(entrenados)} detectores entrenados en {time.time() - t0:.0f} s')

In [ ]:
def metricas(real, pred):
    vp = int((real & pred).sum()); fp = int((~real & pred).sum())
    fn = int((real & ~pred).sum()); vn = int((~real & ~pred).sum())
    prec = vp / (vp + fp) if vp + fp else 0.0
    rec = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {'VP': vp, 'FP': fp, 'FN': fn, 'VN': vn, 'Precisión': prec, 'Recall': rec, 'F1': f1,
            'FPR (%)': 100 * fp / (fp + vn) if fp + vn else 0.0}


filas, por_tipo, por_aparato, ejemplo_pred = [], [], [], None
for rep in range(REPETICIONES):
    datos = inyectar(prueba, 1000 + rep)
    for nombre in MODELOS:
        pred = pd.Series(False, index=datos.index)
        for a in ORDEN:
            sel = datos.aparato == a
            e = entrenados[(a, nombre)]
            Z = e['escala'].transform(datos.loc[sel, VARIABLES_ANOMALIAS].to_numpy(float))
            pred[sel] = e['modelo'].puntaje(Z) > e['umbral']
        real = datos.anomala.to_numpy(bool); p = pred.to_numpy(bool)
        filas.append({'repeticion': rep, 'Modelo': nombre, **metricas(real, p)})
        for t in TIPOS:
            s = (datos.tipo == t).to_numpy()
            if s.any():
                por_tipo.append({'repeticion': rep, 'Modelo': nombre, 'tipo': t, 'recall': p[s].mean()})
        for a in ORDEN:
            s = (datos.aparato == a).to_numpy()
            por_aparato.append({'repeticion': rep, 'Modelo': nombre, 'aparato': a, **metricas(real[s], p[s])})
        if rep == 0 and nombre == 'Isolation Forest':
            ejemplo_pred = datos.assign(alerta=p)

res = pd.DataFrame(filas)
ORDEN_MOD = list(MODELOS)
resumen = res.groupby('Modelo')[['Precisión', 'Recall', 'F1', 'FPR (%)']].agg(['mean', 'std']).reindex(ORDEN_MOD)
tabla_tesis = pd.DataFrame({c: resumen[(c, 'mean')] for c in ['Precisión', 'Recall', 'F1', 'FPR (%)']})
display(Markdown(f'**Resultados en la semana de prueba** (promedio de {REPETICIONES} repeticiones; '
                 'entre paréntesis, la desviación estándar)'))
display(pd.DataFrame({c: [f'{m:.3f} ({s:.3f})' if c != 'FPR (%)' else f'{m:.2f} ({s:.2f})'
                          for m, s in zip(resumen[(c, 'mean')], resumen[(c, 'std')])]
                      for c in ['Precisión', 'Recall', 'F1', 'FPR (%)']}, index=resumen.index))

## 8. Detalle
- Matriz de confusión del Isolation Forest (suma de las 10 repeticiones).
- Recall por tipo de anomalía.
- Resultados del Isolation Forest por aparato.

In [ ]:
mc = res[res.Modelo == 'Isolation Forest'][['VP', 'FP', 'FN', 'VN']].sum()
matriz = pd.DataFrame([[mc.VP, mc.FN], [mc.FP, mc.VN]],
                      index=['Real: anómala', 'Real: normal'], columns=['Detectada: anómala', 'Detectada: normal'])
display(Markdown(f'**Matriz de confusión del Isolation Forest** (suma de {REPETICIONES} repeticiones)'))
display(matriz)

tipo = pd.DataFrame(por_tipo).groupby(['Modelo', 'tipo']).recall.mean().unstack().reindex(ORDEN_MOD)[TIPOS]
display(Markdown('**Recall por tipo de anomalía** (fracción de las horas alteradas que el modelo detectó)'))
display(tipo.round(3))

ap = (pd.DataFrame(por_aparato).query('Modelo == "Isolation Forest"')
        .groupby('aparato')[['Precisión', 'Recall', 'F1', 'FPR (%)']].mean().reindex(ORDEN))
display(Markdown('**Isolation Forest por aparato**'))
display(ap.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(ORDEN_MOD)); w = 0.26
for k, (col, color) in enumerate([('Precisión', '#86b6ef'), ('Recall', '#2a78d6'), ('F1', '#184f95')]):
    axes[0].bar(x + (k - 1) * w, tabla_tesis[col], w, color=color, label=col)
axes[0].axhline(0.85, color=TINTA2, lw=1, ls='--'); axes[0].text(len(x) - 0.5, 0.86, 'F1 ≥ 0,85', ha='right', fontsize=9, color=TINTA2)
axes[0].set_xticks(x); axes[0].set_xticklabels(ORDEN_MOD); axes[0].set_ylim(0, 1.05)
axes[0].legend(ncol=3, loc='upper left'); axes[0].set_title('Precisión, recall y F1')
axes[1].bar(x, tabla_tesis['FPR (%)'], 0.5, color='#2a78d6')
axes[1].axhline(5, color=TINTA2, lw=1, ls='--'); axes[1].text(len(x) - 0.5, 5.2, 'FPR ≤ 5 %', ha='right', fontsize=9, color=TINTA2)
axes[1].set_xticks(x); axes[1].set_xticklabels(ORDEN_MOD); axes[1].set_title('Tasa de falsos positivos (%)')
plt.tight_layout(); plt.savefig('comparacion_detectores.png', dpi=200, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(len(ORDEN), 1, figsize=(14, 11), sharex=True)
for ax, a in zip(axes, ORDEN):
    g = ejemplo_pred[ejemplo_pred.aparato == a]
    ax.plot(g.fecha_hora, g.kwh * 1000, color=COLOR[a], lw=0.9)
    vp = g[g.anomala & g.alerta]; fn = g[g.anomala & ~g.alerta]; fp = g[~g.anomala & g.alerta]
    ax.scatter(vp.fecha_hora, vp.kwh * 1000, s=30, color=TINTA, zorder=3, label='Anomalía detectada')
    ax.scatter(fn.fecha_hora, fn.kwh * 1000, s=30, facecolors='white', edgecolors=TINTA, zorder=3, label='Anomalía no detectada')
    ax.scatter(fp.fecha_hora, fp.kwh * 1000, s=34, marker='x', color='#e34948', zorder=3, label='Falsa alarma')
    ax.text(0.005, 0.82, a, transform=ax.transAxes, fontsize=11, fontweight='bold'); ax.set_ylabel('Wh')
axes[0].legend(loc='upper right', ncol=3)
axes[-1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%d-%b'))
fig.suptitle('Isolation Forest en la semana de prueba con anomalías inyectadas (repetición 1)', x=0.01, ha='left',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('deteccion_semana_prueba.png', dpi=200, bbox_inches='tight'); plt.show()

## 9. Resultados
`resultados_anomalias.zip` contiene las tablas en CSV y las figuras.

In [ ]:
os.makedirs('resultados_anomalias', exist_ok=True)
tabla_tesis.round(4).to_csv('resultados_anomalias/tabla_modelos.csv')
resumen.round(4).to_csv('resultados_anomalias/tabla_modelos_con_desviacion.csv')
matriz.to_csv('resultados_anomalias/matriz_confusion_IF.csv')
tipo.round(4).to_csv('resultados_anomalias/recall_por_tipo.csv')
ap.round(4).to_csv('resultados_anomalias/IF_por_aparato.csv')
res.to_csv('resultados_anomalias/todas_las_repeticiones.csv', index=False)
for f in ('comparacion_detectores.png', 'deteccion_semana_prueba.png'):
    shutil.copy(f, 'resultados_anomalias/')
info = {'semana_prueba': f'{INICIO_PRUEBA:%Y-%m-%d} a {FIN_PRUEBA - pd.Timedelta(days=1):%Y-%m-%d}',
        'horas_entrenamiento': int(len(entreno)), 'horas_prueba': int(len(prueba)),
        'tasa_anomalias': TASA_ANOMALIAS, 'repeticiones': REPETICIONES,
        'percentil_umbral': PERCENTIL_UMBRAL, 'k_vecinos': K_VECINOS, 'grupos_cblof': N_GRUPOS,
        'scikit_learn': sklearn.__version__}
json.dump(info, open('resultados_anomalias/info.json', 'w'), indent=1, ensure_ascii=False)
shutil.make_archive('resultados_anomalias', 'zip', 'resultados_anomalias')
print('Listo: resultados_anomalias.zip')
try:
    from google.colab import files
    files.download('resultados_anomalias.zip')
except ImportError:
    print('No estás en Colab: el zip quedó en la carpeta actual.')